### Load Jar Files

In [1]:
import os
import sys
# 1. Set PYSPARK_SUBMIT_ARGS to match your working batch file launcher
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf spark.driver.extraClassPath="C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar" '
    "pyspark-shell"
)

# 2. Ensure Python paths align for the worker processes
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
from pyspark.sql import SparkSession

BUCKET_NAME = "iceberg"
RPT_WAREHOUSE_PATH = f"s3a://{BUCKET_NAME}/iceberg/WideWorldImportersDW"
WH_CATALOG_NAME = "reporting"

spark = SparkSession.builder \
    .appName("Iceberg Reporting Access") \
    .config("spark.jars.packages", 
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", RPT_WAREHOUSE_PATH) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()

# Query table directly
df_dates = spark.table("reporting.dimension.dates")
df_dates.show(5)

+----------+----+----------+-----+----------+----------------+---+--------------+--------+--------------+-------+------------+-----------+-----------+----------+-------------------+--------------------+---------------+--------------+----------+-------------+------------------+-----------------+-----------------+----------------+
|  date_key|year|year_short|month|month_name|month_name_short|day|formatted_date|day_name|day_name_short|quarter|week_of_year|day_of_week|day_of_year|is_weekend|british_date_format|american_date_format|iso_date_format|yesterday_date|today_date|tomorrow_date|first_day_of_month|last_day_of_month|first_day_of_year|last_day_of_year|
+----------+----+----------+-----+----------+----------------+---+--------------+--------+--------------+-------+------------+-----------+-----------+----------+-------------------+--------------------+---------------+--------------+----------+-------------+------------------+-----------------+-----------------+----------------+
|2026-0

In [8]:
spark.sql("DESCRIBE TABLE EXTENDED reporting.dimension.dates").show(50, truncate=False)

+----------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                                                          |comment|
+----------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|date_key                    |date                                                                                                                                               |NULL   |
|year                        |int                                                                                                                                                |NULL   |
|year_short                  |string                             